**Цель проекта**

Разработать модель, которая предсказывает пол(0 или 1) пользователя на основе его активности в соцсетях.

**Задачи:**

- Загрузка данных и объединение таблиц

- Предобработка данных

- Обучение модели - Catboost

- Оценка качества модели

**Желаемый результат**

Добиться как можно более высокого качества модели(f1-score)


1. Загрузка и объединение таблиц

In [4]:
import pandas as pd
import os

train = pd.read_csv("train(1).csv",sep=";")
test = pd.read_csv("test(1).csv",sep=";")

train_labels = pd.read_csv("train_labels.csv",sep=";")
test_users = pd.read_csv("test_users.csv", sep=";")
referer_vectors = pd.read_csv("referer_vectors.csv",sep=";")
geo_info = pd.read_csv("geo_info.csv",sep=";")

train_merged = train.merge(referer_vectors, on="referer", how="left")
train_merged = train_merged.merge(geo_info, on="geo_id", how="left")
train_merged = train_merged.merge(train_labels, on="user_id", how="inner")

test_merged = test.merge(referer_vectors, on="referer", how="left")
test_merged = test_merged.merge(geo_info, on="geo_id", how="left")


test_merged

,request_ts,user_id,referer,geo_id,user_agent,component0,component1,component2,component3,component4,component5,component6,component7,component8,component9,country_id,region_id,timezone_id
0,1700993094,c2802dadd33d8ae09bb366bdd41212ea,https://9b48ee5/,8816,"{'browser': 'Chrome Mobile', 'browser_version'...",11731,4045,22213,-1184,-8992,9381,-3496,-3120,-899,16817,c31b4e,36e3f3,f6155e
1,1701005579,e5b1988db74527ec092f28b0bbfdaac9,https://9b48ee5/,3663,"{'browser': 'Chrome', 'browser_version': '116....",11731,4045,22213,-1184,-8992,9381,-3496,-3120,-899,16817,c31b4e,8ccc01,e56e80
2,1700969752,6ef1eedbdb72554e53e69782066065c5,https://72879b4/12411b9e,2336,"{'browser': 'Chrome', 'browser_version': '114....",-7307,11682,9741,13564,13577,1200,10169,16461,-3932,3340,c31b4e,1fbfa5,e56e80
3,1700991608,7e057293ecae62985a327b7af51858ea,https://9b48ee5/,9652,"{'browser': 'Chrome Mobile', 'browser_version'...",11731,4045,22213,-1184,-8992,9381,-3496,-3120,-899,16817,c31b4e,f66ff,f6155e
4,1701019815,a27bd7ce8828497823fa8d5d05e7bbf7,https://9b48ee5/,3871,"{'browser': 'Chrome Mobile', 'browser_version'...",11731,4045,22213,-1184,-8992,9381,-3496,-3120,-899,16817,c31b4e,245864,e56e80
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
153078,1700968705,3c86b41e45cc2b7029d969e4e045fc8d,https://8624dd2/,4106,"{'browser': 'Chrome', 'browser_version': '109....",-7489,3356,11800,-9387,20562,10689,13210,-2615,9753,1102,c31b4e,5f9c68,9cdb77
153079,1700989358,8daa3e87a079f857f559062fbf2c02e1,https://72879b4/13cb1eab,4863,"{'browser': 'Yandex Browser', 'browser_version...",8011,11687,1439,13197,5889,-3233,14858,5281,14728,11103,c31b4e,776e76,10b7947
153080,1701025826,8bd30a0a06100a0c782046ec151fcf03,https://6a81948/,1724,"{'browser': 'YandexSearch', 'browser_version':...",10214,-1254,10495,7705,9547,5801,864,8837,5312,22635,c31b4e,1b18a1,e56e80
153081,1700979975,0bba77ee4a9a5f03c7dac143b77a7a0d,https://650870a/142422ab,6436,"{'browser': 'Chrome', 'browser_version': '87.0...",2794,-10182,-18072,-4222,18836,-10290,7087,-2586,17127,-921,c31b4e,8ccc01,e56e80


Заполненим пропущенных значений

In [5]:
train_merged = train_merged.fillna({'region_id': 'none'})
train_merged = train_merged.fillna({'user_agent': 'none'})

test_merged = test_merged.fillna({'region_id': 'none'})

Проверим сколько осталось пропущенных значений

In [6]:
for col in train_merged.columns:
  temp_count = train_merged.loc[train_merged.loc[:, col].isna()].shape[0]
  print(temp_count)

0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0


In [7]:
for col in test_merged.columns:
  temp_count = test_merged.loc[test_merged.loc[:, col].isna()].shape[0]
  print(temp_count)

0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0


In [8]:
!pip install --upgrade user-agents
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.1/86.1 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 5.6 MB/s eta 0:00:00


2. Предобработка данных

Выделяем новые столбцы для времени, домена и информаци об устройствах.

In [9]:
from user_agents import parse
from ast import literal_eval
import json
import ast

def safe_parse(ua_str):
    try:
        return json.loads(ua_str)
    except json.JSONDecodeError:
        try:
            return ast.literal_eval(ua_str)
        except:
            return {'browser': 'Unknown', 'browser_version': "Unknown"}

def extract_ua_info(ua_str):
    ua = parse(ua_str)
    return {
        'browser': ua.browser.family,
        'os': ua.os.family,
        'device_type': 'mobile' if ua.is_mobile else ('tablet' if ua.is_tablet else 'desktop')
    }
for df in [train_merged, test_merged]:
    df['request_ts'] = pd.to_datetime(df['request_ts'], unit='s')
    df['hour'] = df['request_ts'].dt.hour
    df['weekday'] = df['request_ts'].dt.weekday
    df['is_weekend'] = df['weekday'].isin([5, 6]).astype(int)

train_merged['domain'] = train_merged["referer"].apply(lambda elem: elem.replace('https://', '').split('/')[0])
train_merged['domain'] = train_merged["referer"].apply(lambda elem: elem.replace('https://', '').split('/')[0])

train_merged['user_agent'] = train_merged['user_agent'].apply(safe_parse)

train_merged['browser'] = train_merged['user_agent'].apply(lambda x: x.get('browser', 'Unknown'))
train_merged['os'] = train_merged['user_agent'].apply(lambda x: x.get('browser_version', 'Unknown'))

test_merged['domain'] = test_merged["referer"].apply(lambda elem: elem.replace('https://', '').split('/')[0])
test_merged['domain'] = test_merged["referer"].apply(lambda elem: elem.replace('https://', '').split('/')[0])

test_merged['user_agent'] = test_merged['user_agent'].apply(safe_parse)

test_merged['browser'] = test_merged['user_agent'].apply(lambda x: x.get('browser', 'Unknown'))
test_merged['os'] = test_merged['user_agent'].apply(lambda x: x.get('browser_version', 'Unknown'))


train_merged

,request_ts,user_id,referer,geo_id,user_agent,component0,component1,component2,component3,component4,...,country_id,region_id,timezone_id,target,hour,weekday,is_weekend,domain,browser,os
0,2023-11-26 15:09:23,fb858e8e0a2bec074450eaf94b627fd3,https://9b48ee5/,4799,"{'browser': 'Chrome Mobile', 'browser_version'...",11731,4045,22213,-1184,-8992,...,c31b4e,470e75,f6155e,0,15,6,1,9b48ee5,Chrome Mobile,119.0.0
1,2023-11-26 08:16:21,46a5f128fd569c764a92c2eaa788095e,https://9b48ee5/,8257,"{'browser': 'Chrome Mobile', 'browser_version'...",11731,4045,22213,-1184,-8992,...,c31b4e,44520b,e56e80,0,8,6,1,9b48ee5,Chrome Mobile,111.0.0
2,2023-11-26 15:04:31,5a74e9ac53ffb21a20cce117c0ad77ba,https://9634fd0/1409e548,3150,"{'browser': 'Yandex Browser', 'browser_version...",12498,2451,10304,-6380,11608,...,c31b4e,616bb9,af47f1,0,15,6,1,9634fd0,Yandex Browser,20.12.5
3,2023-11-26 10:00:03,af735816ca19115431ae3d89518c8c91,https://9b48ee5/,2740,"{'browser': 'Chrome Mobile', 'browser_version'...",11731,4045,22213,-1184,-8992,...,c31b4e,3c9dca,e56e80,0,10,6,1,9b48ee5,Chrome Mobile,119.0.0
4,2023-11-26 18:01:06,364f0ae0a3f29a685c4fb5bae6033b9a,https://9b48ee5/,4863,"{'browser': 'Yandex Browser', 'browser_version...",11731,4045,22213,-1184,-8992,...,c31b4e,776e76,10b7947,0,18,6,1,9b48ee5,Yandex Browser,18.11.1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
237455,2023-11-26 08:36:34,5a52f2aa721a7eadb4352794cda2bc32,https://b6e7e37/1458ef49,7096,"{'browser': 'Chrome Mobile', 'browser_version'...",14844,1717,11275,9476,6163,...,c31b4e,2dda09,ced83d,0,8,6,1,b6e7e37,Chrome Mobile,119.0.0
237456,2023-11-26 08:01:06,16bbe484480fd1cd2f815204eef7295b,https://9c23da0/,3968,"{'browser': 'Yandex Browser', 'browser_version...",21423,5842,8662,-7650,-2262,...,c31b4e,908bcb,e7021b,1,8,6,1,9c23da0,Yandex Browser,23.7.5
237457,2023-11-26 03:51:23,b05cf740e6d1a7801fb2f54f3ebc4f02,https://b6e7e37/1458ef49,1660,"{'browser': 'Chrome Mobile', 'browser_version'...",14844,1717,11275,9476,6163,...,121db33,4e3184,ac9429,0,3,6,1,b6e7e37,Chrome Mobile,119.0.0
237458,2023-11-26 14:14:04,4f4fbe19e414bf3af96d19e1d321be09,https://b6e7e37/1458ef49,5836,"{'browser': 'Chrome Mobile', 'browser_version'...",14844,1717,11275,9476,6163,...,121db33,672742,ac9429,0,14,6,1,b6e7e37,Chrome Mobile,87.0.4280


In [ ]:
train_merged.shape


(601290, 25)

In [10]:
users_intersection = list(set(train_merged['user_id'].unique()) & set(test_merged['user_id'].unique()))
len(users_intersection)

3291

Заметим, что train и test выборки пересекаются, удалим это пересечение из train выборки

In [11]:
train_merged = train_merged[~train_merged['user_id'].isin(users_intersection)]
users_intersection = list(set(train_merged['user_id'].unique()) & set(test_merged['user_id'].unique()))
len(users_intersection)

0

Делаем предсказание на тренировочной выборке и оцениваем качество модели

In [12]:
import re
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score
from sklearn.metrics import accuracy_score
from sklearn.metrics import f1_score

cat_cols = ['hour', 'domain', 'browser', 'os', 'country_id', 'region_id']
embedding_regex = re.compile('component')
embedding_cols = list(filter(embedding_regex.match, train_merged.columns))
for col in cat_cols:
    le = LabelEncoder()
    train_merged[col] = le.fit_transform(train_merged[col].astype(str))
train_merged['user_id'] = le.fit_transform(train_merged['user_id'].astype(str))


X = train_merged.loc[:, ['user_id'] + cat_cols + embedding_cols]
y = train_merged.loc[:, 'target'].values
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
X_test = test_merged.loc[:, ['user_id'] + cat_cols + embedding_cols]

model = CatBoostClassifier(
    loss_function='Logloss',
    iterations=1000,
    thread_count=-1,
    depth=6,
    boosting_type='Plain',
    early_stopping_rounds=15,
    auto_class_weights='Balanced',
    random_seed=42
)

model.fit(
    X_train.loc[:, cat_cols + embedding_cols],
    y_train,
    cat_features=cat_cols,
    eval_set=(X_val.loc[:, cat_cols + embedding_cols], y_val),
    verbose=100
)

y_train_pred_proba = model.predict_proba(X_train.loc[:, cat_cols + embedding_cols])[:, 1]
y_train_pred = np.where(y_train_pred_proba > 0.5, 1, 0)

train_score = X_train.loc[:, ['user_id']]
train_score['target'] = y_train
train_score['pred_proba'] = y_train_pred_proba
train_score['pred'] = y_train_pred
train_score = train_score.groupby(['user_id'], as_index=False).aggregate(
    {'target': 'max',
     'pred_proba': 'mean',
     'pred': 'max'
     })
print(f"train_accuracy: {np.round(accuracy_score(train_score.target, train_score.pred), 4)}")
print(f"train_f1: {np.round(f1_score(train_score.target, train_score.pred), 4)}")
print(f"train_roc_auc: {np.round(roc_auc_score(train_score.target, train_score.pred_proba), 4)}")


/tmp/ipython-input-1589142411.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_merged[col] = le.fit_transform(train_merged[col].astype(str))
/tmp/ipython-input-1589142411.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_merged[col] = le.fit_transform(train_merged[col].astype(str))
/tmp/ipython-input-1589142411.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the d

Learning rate set to 0.115325
0:	learn: 0.6437904	test: 0.6438267	best: 0.6438267 (0)	total: 425ms	remaining: 7m 4s
100:	learn: 0.4154096	test: 0.4130574	best: 0.4130574 (100)	total: 33.9s	remaining: 5m 1s
200:	learn: 0.4003033	test: 0.3995930	best: 0.3995930 (200)	total: 1m 10s	remaining: 4m 41s
300:	learn: 0.3925960	test: 0.3940253	best: 0.3940253 (300)	total: 1m 44s	remaining: 4m 3s
400:	learn: 0.3867722	test: 0.3906919	best: 0.3906919 (400)	total: 2m 23s	remaining: 3m 33s
500:	learn: 0.3823413	test: 0.3889013	best: 0.3889005 (499)	total: 2m 59s	remaining: 2m 58s
600:	learn: 0.3786008	test: 0.3875989	best: 0.3875947 (598)	total: 3m 34s	remaining: 2m 22s
700:	learn: 0.3753418	test: 0.3865721	best: 0.3865721 (700)	total: 4m 11s	remaining: 1m 47s
800:	learn: 0.3723733	test: 0.3857888	best: 0.3857888 (800)	total: 4m 48s	remaining: 1m 11s
900:	learn: 0.3693448	test: 0.3849232	best: 0.3849171 (894)	total: 5m 23s	remaining: 35.6s
999:	learn: 0.3669249	test: 0.3845085	best: 0.3845085 (999)	

Делаем предсказание на валидационной выборке и оцениваем результат

In [13]:
y_val_pred_proba = model.predict_proba(X_val.loc[:, cat_cols + embedding_cols])[:, 1]
y_val_pred = np.where(y_val_pred_proba > 0.5, 1, 0)
val_score = X_val.loc[:, ['user_id']]
val_score['target'] = y_val
val_score['pred_proba'] = y_val_pred_proba
val_score['pred'] = y_val_pred
val_score = val_score.groupby(['user_id'], as_index=False).aggregate({
    'target': 'max',
    'pred_proba': 'mean',
    'pred': 'max'
})
print(f"val_accuracy: {np.round(accuracy_score(val_score.target, val_score.pred), 4)}")
print(f"val_f1: {np.round(f1_score(val_score.target, val_score.pred), 4)}")
print(f"val_roc_auc: {np.round(roc_auc_score(val_score.target, val_score.pred_proba), 4)}")

val_accuracy: 0.8545
val_f1: 0.8475
val_roc_auc: 0.9036


Делаем предсказание на тестовой выборке

In [14]:
y_test_pred_proba = model.predict_proba(X_test.loc[:, cat_cols +embedding_cols])[:, 1]
y_test_pred = np.where(y_test_pred_proba > 0.5, 1, 0)

test_result = X_test.loc[:, ['user_id']]
test_result['target'] = y_test_pred

test_result


,user_id,target
0,c2802dadd33d8ae09bb366bdd41212ea,0
1,e5b1988db74527ec092f28b0bbfdaac9,0
2,6ef1eedbdb72554e53e69782066065c5,1
3,7e057293ecae62985a327b7af51858ea,0
4,a27bd7ce8828497823fa8d5d05e7bbf7,0
...,...,...
153078,3c86b41e45cc2b7029d969e4e045fc8d,0
153079,8daa3e87a079f857f559062fbf2c02e1,1
153080,8bd30a0a06100a0c782046ec151fcf03,0
153081,0bba77ee4a9a5f03c7dac143b77a7a0d,1


In [15]:
test_result.to_csv("test_labels.csv")